In [3]:
!cd adelic-spectral-zeta && git pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 551 bytes | 551.00 KiB/s, done.
From https://github.com/sneed-and-feed/adelic-spectral-zeta
   bfe62d2..38441e9  v3_research -> origin/v3_research
Updating bfe62d2..38441e9
Fast-forward
 src/ultrametric_v3/surgery.py | 6 +++---
 1 file changed, 3 insertions(+), 3 deletions(-)


In [1]:
!pip install transformers datasets accelerate huggingface_hub

from huggingface_hub import login
# Your token: hf_YOUR_TOKEN_HERE
login("hf_YOUR_TOKEN_HERE")

import os
import sys

# Clone the repo to get the Surgery code

if not os.path.exists('adelic-spectral-zeta'):
    !git clone -b v3_research https://github.com/sneed-and-feed/adelic-spectral-zeta.git
!cd adelic-spectral-zeta && git pull
sys.path.insert(0, os.path.abspath('adelic-spectral-zeta'))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import load_dataset
from src.ultrametric_v3.llama_patcher import inject_surgery
from src.ultrametric_v3.surgery_trainer import TauAnnealingCallback, SurgeryTrainer

model_id = "meta-llama/Meta-Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Download the model and load directly onto the GPU
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")

# Patch the model with Llama Surgery
model = inject_surgery(model)

# Freeze non-surgery parameters
for name, param in model.named_parameters():
    if "router" not in name:
        param.requires_grad = False

import torchvision.io
# Bypass the Hugging Face datasets VideoReader bug
torchvision.io.VideoReader = type("VideoReader", (object,), {})

from transformers import DataCollatorForLanguageModeling

# Setup Dataset
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train[:1%]")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])

# The Data Collator automatically converts to tensors AND creates the 'labels' for causal LM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./surgery_results",
    per_device_train_batch_size=2,
    learning_rate=1e-3,
    max_steps=100,
    logging_steps=10,
    save_steps=50,
)

trainer = SurgeryTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
    callbacks=[TauAnnealingCallback(initial_tau=1.0, min_tau=0.1, decay_steps=100)],
    surgery_lambda_init=0.0,
    surgery_lambda_max=1.0,
    surgery_ramp_steps=50
)

trainer.train()

Cloning into 'adelic-spectral-zeta'...
remote: Enumerating objects: 4344, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 4344 (delta 61), reused 77 (delta 43), pack-reused 4240 (from 2)
Receiving objects: 100% (4344/4344), 33.55 MiB | 18.88 MiB/s, done.
Resolving deltas: 100% (2185/2185), done.
Already up to date.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

Map:   0%|          | 0/367 [00:00<?, ? examples/s]

Step,Training Loss
10,1.460840
20,1.182435
30,1.197002
40,1.470976
50,1.901295
60,1.948479
70,2.310748
80,2.290608
90,2.224722
100,2.028853


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=1.8015959453582764, metrics={'train_runtime': 164.056, 'train_samples_per_second': 1.219, 'train_steps_per_second': 0.61, 'total_flos': 1152756586905600.0, 'train_loss': 1.8015959453582764, 'epoch': 0.5434782608695652})

In [1]:
import os
import sys
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from datasets import load_dataset
from src.ultrametric_v3.llama_patcher import inject_surgery
from src.ultrametric_v3.surgery_trainer import TauAnnealingCallback, SurgeryTrainer

# 1. Load Model & Patch
model_id = "./Llama-3.1-8B-HF"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16, device_map="auto")
model = inject_surgery(model)

# 2. FREEZE LLAMA'S BRAIN!
# We only want to train the 1,024 Router parameters so we don't cause Catastrophic Forgetting!
for name, param in model.named_parameters():
    if "router" not in name:
        param.requires_grad = False

# 3. Load the FULL dataset (not just 1%)
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True).remove_columns(["text"])
tokenized_datasets.set_format("torch")

# 4. The Real Run!
training_args = TrainingArguments(
    output_dir="./surgery_results_final",
    per_device_train_batch_size=2,
    learning_rate=1e-2, # High learning rate is safe because ONLY the routers are training!
    max_steps=1000,     # Run for 1000 steps so the penalty can slowly carve out the sparse topology
    logging_steps=50,
    save_steps=2000,    # Don't freeze the notebook to save checkpoints!
)

trainer = SurgeryTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    callbacks=[TauAnnealingCallback(initial_tau=1.0, min_tau=0.1, decay_steps=1000)],
    surgery_lambda_init=0.0,
    surgery_lambda_max=2.0,  # Turn up the pressure to prune!
    surgery_ramp_steps=500   # Slowly ramp up the penalty over 500 steps
)

trainer.train()

# 5. Run Inference to see the Pruning Results!
model.eval()

total_heads, pruned_heads = 0, 0
for i, layer in enumerate(model.model.layers):
    router_z = layer.self_attn.router.z
    layer_pruned = (router_z > 0).sum().item() # If z > 0, g_h evaluates to 1.0 (Sparse Mask Applied!)
    pruned_heads += layer_pruned
    total_heads += len(router_z)

print(f"\n🎯 Surgery Result: The model successfully carved out a sparse topology, permanently pruning {pruned_heads}/{total_heads} attention heads!")

prompt = "The spectral zeta function of the p-adic numbers is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=60, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)

print("\n--- GENERATION ---")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

ModuleNotFoundError: No module named 'src'